In [24]:
from openai import OpenAI 
import json
import networkx as nx
import ipycytoscape
import pandas as pd
import os
import math
import re
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore", category= DeprecationWarning)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)


In [25]:
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
llm_model_name = "deepseek-ai/DeepSeek-V3"

In [26]:
load_dotenv()

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")


client = OpenAI(
    # Sets the base URL for the API endpoint to the Nebius service.
    base_url=LITELLM_API_BASE,
    # Sets the API key for authentication. Replace with your actual key, preferably loaded from a secure source.
    api_key=NEBIUS_API_KEY
)

In [27]:
llm_temperature = 0.0
llm_max_tokens = 4096

In [28]:
unstructured_text = """
THe beginning of Nations, those excepted of whom sacred Books have spok'n, is to this day unknown. Nor only the beginning, but the deeds also of many succeeding Ages, yea periods of Ages, either wholly unknown, or obscur'd and blemisht with Fables. Whether it were that the use of Letters came in long after, or were it the violence of barbarous inundations, or they themselves at certain revolutions of time, fatally decaying, and degenerating into Sloth and Ignorance; wherby the monuments of more ancient civility have bin som destroy'd, som lost. Perhaps dis-esteem and contempt of the public affairs then present, as not worth recording, might partly be in cause. Certainly oft-times we see that wise men, and of best abilitie, have forborn to write the Acts of thir own daies, while they beheld with a just loathing and disdain, not only how unworthy, how pervers, how corrupt, but often how ignoble, how petty, how below all History the persons and thir actions were; who either by fortune, or som rude election had attain'd as a sore judgment and ignominie upon the Land, to have cheif sway in managing the Commonwealth. But that any law, or superstition of our old Philosophers the Druids forbad the Britans to write thir memorable deeds, I know not why any out of Cæsar should allege: he indeed saith, that thir doctrine they thought not lawful to commit to Letters; but in most matters else, both privat, and public, among which well may History be reck'nd, they us'd the Greek Tongue: and that the British Druids who taught those in Gaule would be ignorant of any Language known and us'd by thir Disciples, or so frequently writing other things, and so inquisitive into highest, would for want of recording be ever Children in the Knowledge of Times and Ages, is not likely. What ever might be the reason, this we find, that of British affairs, from the first peopling of the Iland to the coming of Julius Cæsar, nothing certain, either by Tradition, History, or Ancient Fame hath hitherto bin left us. That which we have of oldest seeming, hath by the greater part of judicious Antiquaries bin long rejected for a modern Fable.

Nevertheless there being others, besides the first suppos'd Author, men not unread, nor unlerned in Antiquitie, who admitt that for approved story, which the former explode for fiction, and seeing that oft-times relations heertofore accounted fabulous have bin after found to contain in them many footsteps, and reliques of somthing true, as what we read in Poets of the Flood, and Giants little beleev'd, till undoubted witnesses taught us, that all was not fain'd; I have therfore determin'd to bestow the telling over ev'n of these reputed Tales; be it for nothing else but in favour of our English Poets, and Rhetoricians, who by thir Art will know, how to use them judiciously.

I might also produce example, as Diodorus among the Greeks, Livie and others of the Latines, Polydore and Virunnius accounted among our own Writers. But I intend not with controversies and quotations to delay or interrupt the smooth course of History; much less to argue and debate long who were the first Inhabitants, with what probabilities, what authorities each opinion hath bin upheld, but shall endevor that which hitherto hath bin needed most, with plain, and lightsom brevity, to relate well and orderly things worth the noting, so as may best instruct and benefit them that read. Which, imploring divine assistance, that it may redound to his glory, and the good of the British Nation, I now begin.

That the whole Earth was inhabited before the Flood, and to the utmost point of habitable ground, from those effectual words of God in the Creation, may be more then conjectur'd. Hence that this Iland also had her dwellers, her affairs, and perhaps her stories, eev'n in that old World those many hunderd years, with much reason we may inferr. After the Flood, and the dispersing of Nations, as they journey'd leasurely from the East, Gomer the eldest Son of Japhet, and his off-spring, as by Authorities, Arguments, and Affinitie of divers names is generally beleev'd, were the first that peopl'd all these West and Northren Climes. But they of our own Writers, who thought they had don nothing, unless with all circumstance they tell us when, and who first set foot upon this Iland, presume to name out of fabulous and counterfet Authors a certain Samothes or Dis, a fowrth or sixt Son of Japhet, whom they make about 200 years after the Flood, to have planted with Colonies, first the Continent of Celtica or Gaule, and next this Iland; Thence to have nam'd it Samothea, to have reign'd heer, and after him lineally fowr Kings, Magus, Saron, Druis, and Bardus. But the forg'd Berosus, whom only they have to cite, no where mentions that either hee, or any of those whom they bring, did ever pass into Britain, or send thir people hither. So that this outlandish figment may easily excuse our not allowing it the room heer so much as of a British Fable.

That which follows, perhaps as wide from truth, though seeming less impertinent, is, that these Samotheans under the Reign of Bardus were subdu'd by Albion a Giant, Son of Neptune; who call'd the Iland after his own name, and rul'd it 44 years. Till at length passing over into Gaul, in aid of his Brother Lestrygon, against whom Hercules was hasting out of Spain into Italy, he was there slain in fight, and Bergion also his Brother.

"""

In [30]:
char_count = len(unstructured_text)
word_count = len(unstructured_text.split())

print("Word count", word_count)
print("Char count", char_count)

print("Unstructured Text \n", unstructured_text)

Word count 934
Char count 5429
Unstructured Text 
 
THe beginning of Nations, those excepted of whom sacred Books have spok'n, is to this day unknown. Nor only the beginning, but the deeds also of many succeeding Ages, yea periods of Ages, either wholly unknown, or obscur'd and blemisht with Fables. Whether it were that the use of Letters came in long after, or were it the violence of barbarous inundations, or they themselves at certain revolutions of time, fatally decaying, and degenerating into Sloth and Ignorance; wherby the monuments of more ancient civility have bin som destroy'd, som lost. Perhaps dis-esteem and contempt of the public affairs then present, as not worth recording, might partly be in cause. Certainly oft-times we see that wise men, and of best abilitie, have forborn to write the Acts of thir own daies, while they beheld with a just loathing and disdain, not only how unworthy, how pervers, how corrupt, but often how ignoble, how petty, how below all History the pers

In [ ]:
#Chunking Configuration

chunk_size = 150
chunk_overlap = 30

if chunk_overlap > chunk_size and chunk_size > 0:
    raise SystemExit(f"Erorr: Overlap {chunk_overlap} must be smaller than chunk size {chunk_size}")
else:
    print("Chunnking condig is valid")